# 05 — Deployment: Batch Transform scoring of the production cohort

**Team Update 4.1 step 4** (deploy to SageMaker — batch process) and the
scoring half of the design doc's batch architecture.

Scores the reserved **40% production cohort** (33,048 members - labels exist
but stay unused here; notebook 06 feeds them to the quality monitor) with the
registered champion via SageMaker Batch Transform. Output: per-member
probabilities + predicted tier + a ranked High-Risk worklist.

| Step | Verify (📸) |
|---|---|
| Scoring input built with the exact training encoding | schema check print |
| Batch transform job + output | completed-job describe |
| Ranked High-Risk worklist | head() below |

## 1. Setup

In [ ]:
import os
from pathlib import Path

# CWD depends on the runner: Studio file view starts at the repo root, while
# `jupyter nbconvert` starts in the notebook's directory. Resolve the repo
# root (the dir that contains src/) and work from there.
if (Path.cwd() / "src").is_dir():
    REPO = Path.cwd()
elif (Path.cwd().parent / "src").is_dir():
    REPO = Path.cwd().parent
else:  # fresh Studio file view: clone the repo next to the notebooks dir
    !git clone --quiet https://github.com/University-San-Diego-MAAI/AAI-540-Group-1-Final-Project.git ../repo
    REPO = Path.cwd().parent / "repo"
os.chdir(REPO)
print("repo root:", REPO, "| src/train.py:", (REPO / "src" / "train.py").is_file())

In [ ]:
%pip install -q "sagemaker<3"

import json
import time

import boto3
import pandas as pd
import sagemaker

region = boto3.Session().region_name
boto_session = boto3.Session(region_name=region)
sm_client = boto_session.client("sagemaker", region_name=region)
s3 = boto3.client("s3", region_name=region)
bucket = sagemaker.Session(boto_session=boto_session).default_bucket()
prefix = "aai-540-g1"

try:
    role = sagemaker.get_execution_role()
except Exception:
    iam = boto3.client("iam", region_name=region)
    by_name = {r["RoleName"]: r for r in iam.list_roles()["Roles"]}
    if "aai-540-g1-execution-role" in by_name:
        picked = "aai-540-g1-execution-role"
    else:
        cands = [n for n in by_name if n.startswith("AmazonSageMaker-ExecutionRole")]
        picked = sorted(cands, key=lambda n: by_name[n]["CreateDate"])[-1]
    account = boto3.client("sts").get_caller_identity()["Account"]
    role = f"arn:aws:iam::{account}:role/{picked}"

RISK_CLASSES = ["Low", "Medium", "High"]
print(f"role={role}\nbucket={bucket}")

## 2. Build the scoring input from the production split

Encoding must be identical to training: same `encode_features` call on the
full table (fixes column order), then keep production rows. The CSV carries
`person_id` in column 0 so the transform can join it back onto predictions
(exactly the input/output-filter pattern from Assignment 4.1).

In [ ]:
import sys
sys.path.insert(0, "src")
from model_common import encode_features, load_table

s3.download_file(bucket, f"{prefix}/processed/training/training_table.csv",
                 "training_table.csv")
table = load_table("training_table.csv")
X_all, feature_names, audit = encode_features(table)
assert not audit["split"].isna().any()

prod_mask = (audit["split"] == "production").to_numpy()
prod_X = pd.DataFrame(X_all[prod_mask], columns=feature_names)
prod_ids = audit.loc[prod_mask, "person_id"].astype(int).reset_index(drop=True)
scoring = pd.concat([prod_ids, prod_X], axis=1)
scoring.to_csv("scoring_input.csv", index=False, header=False)
print(f"scoring input: {scoring.shape[0]:,} rows x {scoring.shape[1]} cols "
      f"(id + {len(feature_names)} features); headerless as the XGBoost "
      f"container expects")
scoring_input_s3 = f"s3://{bucket}/{prefix}/production/scoring_input.csv"
s3.upload_file("scoring_input.csv", bucket, f"{prefix}/production/scoring_input.csv")

## 3. Create a SageMaker Model from the registered champion

In [ ]:
group = sm_client.list_model_packages(ModelPackageGroupName="member-utilization-risk")
package = sm_client.describe_model_package(
    ModelPackageName=sorted(
        group["ModelPackageSummaryList"],
        key=lambda p: p["ModelPackageVersion"])[-1]["ModelPackageArn"])
# Human-in-the-loop was exercised at registration (PendingManualApproval);
# approve programmatically to deploy the batch scorer (design doc: approval
# gates PRODUCTION promotion; scoring the reserved cohort is the dev deploy).
if package["ModelApprovalStatus"] != "Approved":
    sm_client.update_model_package(
        ModelPackageArn=package["ModelPackageArn"], ModelApprovalStatus="Approved")
    package["ModelApprovalStatus"] = "Approved"
    print("approved package for the batch-scoring deploy")

spec = package["InferenceSpecification"]["Containers"][0]
MODEL_NAME = "member-utilization-risk-champion"
try:
    sm_client.delete_model(ModelName=MODEL_NAME)
except sm_client.exceptions.ClientError:
    pass
sm_client.create_model(
    ModelName=MODEL_NAME,
    ExecutionRoleArn=role,
    PrimaryContainer={"Image": spec["Image"], "ModelDataUrl": spec["ModelDataUrl"]},
)
print(f"model {MODEL_NAME} ready")

## 4. Batch transform (accept/assemble_with set explicitly — HW 4.1 lesson)

`input_filter="$[1:]"` drops the id column before inference;
`join_source="Input"` associates predictions with input rows;
`output_filter="$[0,-1]"` keeps only id + the probability triple.

In [ ]:
JOB = f"score-production-{int(time.time())}"
OUTPUT = f"s3://{bucket}/{prefix}/artifacts/scoring/{JOB}"
sm_client.create_transform_job(
    TransformJobName=JOB,
    ModelName=MODEL_NAME,
    TransformInput={
        "DataSource": {"S3DataSource": {"S3DataType": "S3Prefix",
                                        "S3Uri": scoring_input_s3}},
        "ContentType": "text/csv",
        "SplitType": "Line",
    },
    TransformOutput={"S3OutputPath": OUTPUT, "Accept": "text/csv",
                     "AssembleWith": "Line"},
    TransformResources={"InstanceType": "ml.m5.xlarge", "InstanceCount": 1},
    DataProcessing={
        "InputFilter": "$[1:]",
        "OutputFilter": "$[0,-1]",
        "JoinSource": "Input",
    },
)

while True:
    tj = sm_client.describe_transform_job(TransformJobName=JOB)
    status = tj["TransformJobStatus"]
    if status == "Completed":
        print(f"{JOB} completed in {tj['TransformEndTime'] - tj['TransformStartTime']}")
        break
    if status == "Failed":
        raise RuntimeError(tj.get("FailureReason"))
    time.sleep(15)
print(OUTPUT)

## 5. Parse output → scored cohort + ranked worklist (📸)

With `output_filter="$[0,-1]"` each line is `person_id,<prob triple>` — the
multi:softprob container emits the three class probabilities space-separated
in a single field. Split them back out, argmax to a tier, persist, and build
the ranked worklist.

In [ ]:
import io

out_key = f"{prefix}/artifacts/scoring/{JOB}/scoring_input.csv.out"
raw = s3.get_object(Bucket=bucket, Key=out_key)["Body"].read().decode()
scores = pd.read_csv(io.StringIO(raw), names=["person_id", "proba_string"])
assert len(scores) == len(scoring), \
    f"row count mismatch: {len(scores)} scores vs {len(scoring)} inputs"

proba_cols = ["proba_low", "proba_medium", "proba_high"]
probs = scores["proba_string"].str.split(expand=True).astype(float)
probs.columns = proba_cols
scores = pd.concat([scores[["person_id"]], probs], axis=1)
scores["predicted_risk"] = [
    RISK_CLASSES[i] for i in scores[proba_cols].to_numpy().argmax(axis=1)]

scores.to_csv("production_scores.csv", index=False)
s3.upload_file("production_scores.csv", bucket,
               f"{prefix}/artifacts/scoring/production_scores.csv")
print(f"scored {len(scores):,} members; predicted distribution:",
      scores["predicted_risk"].value_counts(normalize=True).round(3).to_dict())

worklist = scores.sort_values("proba_high", ascending=False).reset_index(drop=True)
worklist.index.name = "priority_rank"
worklist.head(10)